# **Data Extraction**

In [ ]:
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


In [ ]:
# Step 1: Send an HTTP request
url = 'https://onepiece.fandom.com/wiki/Devil_Fruit'
response = requests.get(url)
content = response.content

In [ ]:
# Step 2: Parse the HTML content
soup = BeautifulSoup(content, 'html.parser')

In [ ]:
# Step 3: Locate the navbox table
navbox_table = soup.find('table', {'class': 'navibox'})

# Initialize lists to store Devil Fruit names and URLs
fruit_names = []
urls = []

if navbox_table:
    # Find all the rows in the table
    rows = navbox_table.find_all('tr')

    for row in rows:
        # Find all <a> tags in the row
        links = row.find_all('a', href=True)
        for link in links:
            # Extract the text and URL
            fruit_name = link.get_text(strip=True)
            url = link['href']

            # If URL is relative, prepend the base URL
            if url.startswith('/'):
                url = 'https://onepiece.fandom.com' + url

            # Append to lists
            if fruit_name.endswith("Mi"):
                fruit_names.append(fruit_name)
                urls.append(url)


In [ ]:
fruit_names

In [ ]:
urls

In [ ]:
# Step 4: Create a DataFrame
df = pd.DataFrame({
    'Devil Fruit': fruit_names,
    'URL': urls
})


In [ ]:
df[df.duplicated()]

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
# Save the scraped list in the processed data folder
df.to_csv(PROCESSED_DATA_DIR / "devil_fruits.csv", index=False)


In [ ]:
# now we use octoparse to extract info about fruits

In [ ]:
df = pd.read_csv(RAW_DATA_DIR / "devil_fruits_info.csv")


In [ ]:
df.drop(['Text5','Text1','Text6','Text3', 'Text4','Text2'],axis=1, inplace=True)

In [ ]:
df[df.isnull().any(axis=1)]
df.fillna('', inplace=True)

In [ ]:
df.rename(columns={'Text': 'Meaning'}, inplace=True)

In [ ]:
df.Description

In [ ]:
df.to_csv(PROCESSED_DATA_DIR / "devil_fruits_info.csv", index=False)


In [ ]:
df = pd.read_csv(PROCESSED_DATA_DIR / "devil_fruits_info.csv")


In [ ]:
df.head()

# **Characters Network**

In [ ]:
import pandas as pd

In [ ]:
subtitles_paths = sorted((RAW_DATA_DIR / "subtitles").glob("*.ass"))


In [ ]:
scripts = []
episode_num = []
for path in subtitles_paths:
    with path.open("r", encoding="utf-8") as file:
        lines = file.readlines()[27:]
        rows = [",".join(line.split(",")[9:]) for line in lines]
    rows = [line.replace("\\N", " ") for line in rows]
    script = " ".join(rows)
    episode = int(path.stem)

    scripts.append(script)
    episode_num.append(episode)


In [ ]:
df = pd.DataFrame.from_dict({'Episode': episode_num, "Script": scripts})

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
from nltk import sent_tokenize

In [ ]:
df

In [ ]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import clean_episode_scripts

df["Script"] = clean_episode_scripts(df["Script"])


In [ ]:
df

In [ ]:
def get_ners(script):
    script_sentences = sent_tokenize(script)
    ner_output = []
    for sentence in script_sentences:
        doc = nlp(sentence)
        ners = set()
        for entity in doc.ents:
            if entity.label_ == "PERSON":
                full_name = entity.text
                first_name = full_name.split(" ")[0]
                ners.add(first_name)
        ner_output.append(list(ners))
    return ner_output

In [ ]:
df['ners'] = df['Script'].apply(get_ners)

In [ ]:
df.to_csv(PROCESSED_DATA_DIR / "character_network.csv", index=False)


In [ ]:
df = pd.read_csv(PROCESSED_DATA_DIR / "character_network.csv")
import ast

df["ners"] = df["ners"].apply(ast.literal_eval)

In [ ]:
df

In [ ]:
df.iloc[3,2]

In [ ]:
# for each time the two characters appear within 10 sentences, we add them to a list
window = 10
entity_relationships = []

for row in df['ners']:
    previous_entities_in_window = []
    for sentence in row:
        previous_entities_in_window.append(sentence)
        # Keep only the last 10 sentences in the list
        previous_entities_in_window = previous_entities_in_window[-10:]
        previous_entities_in_window_flatten = sum(previous_entities_in_window, [])

        for entity in sentence:
            for entity_in_window in previous_entities_in_window_flatten:
                if entity != entity_in_window:
                    entity_relation = sorted([entity, entity_in_window])
                    entity_relationships.append(entity_relation)



In [ ]:
relationships_df = pd.DataFrame({"value":entity_relationships})

In [ ]:
relationships_df['source'] = relationships_df['value'].apply(lambda x: x[0])
relationships_df['target'] = relationships_df['value'].apply(lambda x: x[1])

relationships_df = relationships_df.groupby(["source", "target"]).count().reset_index()

relationships_df = relationships_df.sort_values("value", ascending = False)
relationships_df.rename(columns={'value': 'count'}, inplace=True)

In [ ]:
relationships_df = relationships_df.head(200)
# we want top 200 most important connections between characters

In [ ]:
relationships_df

In [ ]:
import networkx as nx
graph = nx.from_pandas_edgelist( relationships_df,source = "source",target = "target",edge_attr = "count",create_using = nx.Graph())

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
pos = nx.kamada_kawai_layout(graph)
nx.draw(graph, with_labels=True, node_color="pink", edge_cmap=plt.cm.Blues, pos=pos)
plt.show()


In [ ]:
# 3D visualization

from IPython.display import HTML, display
from pyvis.network import Network

node_degree = dict(graph.degree)
nx.set_node_attributes(graph, node_degree, "size")
node_label = {node: node for node in graph.nodes()}
nx.set_node_attributes(graph, node_label, "label")

net = Network(
    notebook=True,
    width="1000px",
    height="700px",
    bgcolor="#222222",
    font_color="white",
    cdn_resources="in_line",
)
net.from_nx(graph)
net.set_options("""
var options = {
  "nodes": {
    "font": {
      "size": 16,
      "color": "white"
    }
  },
  "interaction": {
    "hover": true,
    "zoomView": true
  },
  "physics": {
    "enabled": true
  }
}
""")

display(HTML(net.generate_html()))


In [ ]:
'''
issues i have noticed:
1- names of cities, ships in one piece world are considered characters
3- L-Luffy == Luffy (character names with sounds added to imphasis a feeling or action are considered different characters)
6- some characters have unique names not being considered as entities, such as mr. one, miss two,...
7- some names are not clean
8- curse / bad words
9- adjectives are considered character names for some reason!!

solution is more data cleaning!----> needs domain knowledge
'''


# **Theme Classifier**

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(PROCESSED_DATA_DIR / "character_network.csv")
df.head()


In [73]:
import torch
print(torch.__version__)

2.14.1+cu130


In [74]:
import sys
print(sys.executable)


/home/heba/heba-backup/OnePieceDataAnalysis/.venv/bin/python


In [75]:
device = 0 if torch.cuda.is_available() else 'cpu'

In [ ]:
import os

from huggingface_hub import InferenceClient

MODEL_ID = "facebook/bart-large-mnli"
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "Set HF_TOKEN to a Hugging Face token with Inference Providers access."
    )

inference_client = InferenceClient(
    model=MODEL_ID,
    token=HF_TOKEN,
    timeout=60,
)


In [77]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/heba/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/heba/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/heba/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [78]:
theme_classifier

ZeroShotClassificationPipeline: {'model': 'BartForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [79]:
df

,Episode,Script,ners
0,1,"ONE PIECE! We are! Whoa... ""I'm Luffy! The Man...","[[], [], [], [], [], [], [], [], [], [], [], [..."
1,2,ONE PIECE! We are! What great weather! If we k...,"[[], [], [], [], [], [], [], [], [], [], [], [..."
2,3,ONE PIECE! We are! Fire! What are you--?! Luff...,"[[], [], [], [], [], [Straw], [], [], [], [], ..."
3,4,ONE PIECE! We are! Another big haul today! Bug...,"[[], [], [], [Buggy], [], [], [], [], [], [], ..."
4,5,ONE PIECE! We are! Get back here! Give that ch...,"[[], [], [], [], [], [], [], [], [], [Buggy], ..."
...,...,...,...
195,265,Text Luffy and the gang have arrived in Enies ...,"[[Text], [Franky, Nico], [], [Robin], [], [], ..."
196,266,"Text So, you guys stay and wait on the sea for...","[[], [], [], [], [], [], [], [], [], [], [ya],..."
197,267,Text We're not gonna let even an ant pass thro...,"[[], [Kashii], [], [], [], [], [], [], [], [],..."
198,268,Text Franky and Robin have been taken to the T...,"[[Robin, Text], [], [Cutty], [], [], [], [Tom]..."


In [ ]:
from nltk.tokenize import sent_tokenize

CANDIDATE_LABELS = [
    "friendship", "hope", "sacrifice", "fight", "betrayal", "hero","dream", "sadness", "anger", "murder", "evil",
]


def get_themes(script):
    """Classify an episode script through Hugging Face Inference Providers."""
    if not isinstance(script, str) or not script.strip():
        return {label: 0.0 for label in CANDIDATE_LABELS}

    sentences = sent_tokenize(script)
    chunks = [
        " ".join(sentences[index : index + 20])
        for index in range(0, len(sentences), 20)
    ]
    if not chunks:
        return {label: 0.0 for label in CANDIDATE_LABELS}

    scores_by_label = {label: [] for label in CANDIDATE_LABELS}
    for chunk in chunks:
        result = inference_client.zero_shot_classification(
            chunk,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=True,
        )
        for item in result:
            scores_by_label[item.label].append(item.score)

    return {
        label: float(np.mean(scores))
        for label, scores in scores_by_label.items()
        if scores
    }


In [81]:
import numpy as np

In [83]:
df['Script'][0]

'ONE PIECE! We are! Whoa... "I\'m Luffy! The Man Who\'s Gonna be King of the Pirates!" No worries! A whirlpool of that size poses no problem to this ship! Miss. May I have this dance? Alright! Not again... Batter out! What\'re you doing? You suck! This thing\'s heavy! Must be filled with booze, then! It\'s all ours now! Ship off the starboard stern! There\'s a pirate flag on its mast! It\'s a pirate ship! Enemy raid! Enemy raid! Captain! Pirates! What?! Everyone, stay calm! Stay calm! Follow instructions in a calm manner! Fire! Coby! Y-Yes, ma\'am?! Who\'s the most beautiful of all on these seas? T-This ship\'s captain, Alvida, of course! Well done! T-Thank you! Alvida. This is our first prey in a while. That\'s right! Alright! Start pounding the ship! Men! Show them the terror of Iron Club Alvida, the woman pirate! Coby! What\'re you doing?! I-I\'m not good at this sort of thing... Hmm?! Are you tryin\' to give me backtalk?! P-Please, anything but the club! Then get yer ass in gear! W

In [82]:
get_themes(df['Script'][0])

KeyboardInterrupt: 

In [ ]:
output_emotions = df['Script'].apply(get_themes)

In [ ]:
output_emotions[0]

In [ ]:
# sequence_to_classify = "one day I will see the world"
# candidate_labels = ['travel', 'cooking', 'dancing']
# classifier(sequence_to_classify, candidate_labels)

In [ ]:
# # pose sequence as a NLI premise and label as a hypothesis
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# nli_model = AutoModelForSequenceClassification.from_pretrained('facebook/bart-large-mnli')
# tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-mnli')

# premise = sequence
# hypothesis = f'This example is {label}.'

# # run through model pre-trained on MNLI
# x = tokenizer.encode(premise, hypothesis, return_tensors='pt',
#                      truncation_strategy='only_first')
# logits = nli_model(x.to(device))[0]

# # we throw away "neutral" (dim 1) and take the probability of
# # "entailment" (2) as the probability of the label being true
# entail_contradiction_logits = logits[:,[0,2]]
# probs = entail_contradiction_logits.softmax(dim=1)
# prob_label_is_true = probs[:,1]